# Deforestation v2: full run on Colab
Runtime → Change runtime type → **T4 GPU**. Results are saved to Google Drive, so a disconnect only loses the run in progress. Re-run all cells to resume.

In [1]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/deforestation-v2'
!mkdir -p $WORK

GPU 0: Tesla T4 (UUID: GPU-5cbf56ea-ef1f-c6de-92e5-898c21d072a5)
Mounted at /content/drive


In [2]:
%cd /content
!rm -rf deforestation-prediction-india
!git clone -q https://github.com/kavya-sajjit/deforestation-prediction-india.git
%cd deforestation-prediction-india
!pip install -q -r requirements.txt

/content
/content/deforestation-prediction-india
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Quick tests (about 1 minute)
The slow end-to-end test is skipped here; it already passes in the repo.

In [ ]:
!pytest -q tests/ -k "not end_to_end"

..

## 2. Build the dataset (Sentinel-2 + Hansen), about 10–20 min per region
Resumable: regions that finished are skipped if Colab disconnects. Just re-run.

In [ ]:
!python -m deforest.data.build --out $WORK/data/v2
!cat $WORK/data/v2/dataset_summary.json

Check the class counts above. If a region has fewer than ~40 forecast positives, move its box with `scripts/scan_hansen_aoi.py`, edit `deforest/config.py`, delete that region's files and rebuild it with `--regions <name>`.

## 3. FAST first results (about 1–2 h on a T4)
11 priority experiments × 3 spatial folds, 12 epochs. Results save to Drive after every run, so a disconnect loses only the run in progress. Re-run the cell to continue.

In [ ]:
  !python -m deforest.experiments --data $WORK/data/v2 --out $WORK/results --profile core --only priority

## 4. Optional, run later: leave-one-region-out and the remaining folds
Only after section 3 has finished. Not needed before you apply.

In [ ]:
# !python -m deforest.experiments --data $WORK/data/v2 --out $WORK/results --profile fast --scheme loro --only history_lr gbm bcos_rgbn_hist cnn_rgbn_hist
# !python -m deforest.experiments --data $WORK/data/v2 --out $WORK/results --profile fast --scheme block --folds fold3 fold4 --only priority

## 5. Risk maps and summary

In [ ]:
# Risk maps use the fold-0 forecast model. With leave-one-region-out models (section 4), point --model at forecast/loro/... instead.
for r in ['western_ghats','odisha_mining','northeast_hills','central_india']:
    !python -m deforest.risk_map --data $WORK/data/v2 --region {r} --model $WORK/results/forecast/block/bcos_rgbn_hist/fold0/model.pt --out $WORK/maps

In [ ]:
!python -m deforest.summarize $WORK/results
!rm -rf results maps && cp -r $WORK/results results && cp -r $WORK/maps maps
!python scripts/update_readme.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/SUMMARY.md').read()))
import glob
for f in sorted(glob.glob('maps/*.png'))+sorted(glob.glob('results/*/block/*/fold0/gallery.png')):
    print(f); display(Image(f))

## 6. Export data for the website
Re-reads Hansen once per region (yearly loss 2001–2023) and writes the map data, the results summary and the explanation images into `web/public/data`. Download the zip and unzip it into your repo's `web/public/` folder, then commit.

In [ ]:
!python -m deforest.web_export --data $WORK/data/v2 --results $WORK/results --out web/public/data
!cd web/public && zip -qr /content/website_data.zip data img
from google.colab import files
files.download('/content/website_data.zip')

## 7. Commit the results
Download `results/` (without `model.pt` files), `maps/*.png` and `README.md` and commit them, together with the website data from step 6.